<a href="https://colab.research.google.com/github/ravindravala/AI/blob/main/cnn_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [10]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3)
        self.pool1 = nn.MaxPool2d(2,2)
        self.linear = nn.Linear(in_features=8*13*13, out_features=10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool1(x)
        x = torch.flatten(x, 1)
        x = self.linear(x)
        return x




In [11]:
model = SimpleCNN()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

In [12]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# =========================
# 2. DATASET
# =========================
transform = transforms.ToTensor()

train_data = datasets.MNIST(
    root="./data",
    train=True,
    transform=transform,
    download=True
)

test_data = datasets.MNIST(
    root="./data",
    train=False,
    transform=transform
)


In [13]:
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32)

In [14]:
epochs = 10

for epoch in range(epochs):
    total_loss = 0

    for images, labels in train_loader:
        outputs = model(images)
        loss = loss_fn(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 529.1254
Epoch 2, Loss: 190.1700
Epoch 3, Loss: 143.0366
Epoch 4, Loss: 120.9267
Epoch 5, Loss: 107.7674
Epoch 6, Loss: 97.1236
Epoch 7, Loss: 89.0950
Epoch 8, Loss: 81.0565
Epoch 9, Loss: 76.0906
Epoch 10, Loss: 70.8303


In [15]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 98.03%
